# AIoT Project - Feature Engineering

This notebook implements the engineered-feature approach: extract statistical and frequency-domain features from windows, compare all features against selected features, and optionally evaluate combined wrist/chest/ankle sensing.


In [ ]:
import os
import sys
from pathlib import Path

# Set random seeds for reproducibility
import random
import numpy as np
np.random.seed(42)
random.seed(42)

# basic data engineering
import pandas as pd
pd.options.mode.copy_on_write = True
import scipy

# plotting
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.dpi'] = 100

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# db
import pymongo

# configs & other
import yaml
from tqdm.notebook import tqdm_notebook
from datetime import datetime
from time import time
import logging

from psynlig import pca_explained_variance_bar

# utils processing
from utils import sliding_window_pd
from utils import apply_filter
from utils import filter_instances
from utils import flatten_instances_df
from utils import df_rebase
from utils import rename_df_column_values

# utils visualization
from utils_visual import plot_instance_time_domain
from utils_visual import plot_instance_3d
from utils_visual import plot_np_instance
from utils_visual import plot_heatmap
from utils_visual import plot_scatter_pca

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

%load_ext autoreload
%autoreload 2

Start time of execution

In [ ]:
time_start = time()

## Load configuration

In [ ]:
config_path = os.path.join(os.getcwd(), "config.yml")

if not os.path.exists(config_path):
    logger.error(f"Config file not found at {config_path}")
    logger.info("Copy config.yml.template to config.yml and fill in your settings")
    raise FileNotFoundError(config_path)

try:
    with open(config_path) as file:
        config = yaml.load(file, Loader=yaml.FullLoader)
    logger.info(f"Configuration loaded successfully from {config_path}")
except yaml.YAMLError as e:
    logger.error(f"Failed to parse config.yml: {e}")
    raise

# Validate required config keys
required_keys = ["client", "db", "col", "sliding_window", "filter", "classifier", "fine_tune"]
missing_keys = [key for key in required_keys if key not in config]
if missing_keys:
    logger.error(f"Missing required config keys: {missing_keys}")
    raise KeyError(f"Config missing keys: {missing_keys}")

logger.info("Configuration validation passed")

In [ ]:
try:
    client = pymongo.MongoClient(config["client"], serverSelectionTimeoutMS=5000)
    # Verify connection
    client.admin.command("ping")
    logger.info(f"✓ Connected to MongoDB at {config['client']}")
except pymongo.errors.ConnectionFailure as e:
    logger.error(f"✗ Failed to connect to MongoDB: {e}")
    logger.info("Ensure MongoDB is running: mongod or mongodump service")
    raise
except Exception as e:
    logger.error(f"✗ Unexpected error connecting to MongoDB: {e}")
    raise

In [ ]:
db = client[config["db"]]
coll = db[config["col"]]

## Load data

Fetch the documents that correspond to the **sensor configuration you are evaluating**. Start with the **Protocol** split and the hand/wrist IMU in the AccGyr configuration.

```python
query = {"split": "Protocol", "imu_location": "hand", "sensor": "AccGyr"}
documents = list(coll.find(query))
```

The Optional split is useful for extra EDA, but several optional activities appear only for a subset of subjects. For the baseline model, Protocol is cleaner because every Protocol class has subject-disjoint train/test coverage.

Carry the `subject` field through every transformation (windowing, filtering, feature extraction) because the train/test split must be subject-disjoint.

In [ ]:
found_labels = coll.distinct("activity_label", {"split": "Protocol"})
print("Protocol activities in DB:", sorted(found_labels))

In [ ]:
query = {"split": "Protocol", "imu_location": "hand", "sensor": "AccGyr"}
documents = list(coll.find(query))

In [ ]:
print(f"Loaded documents: {len(documents)}")

if len(documents) > 0:
    example = documents[0]
    print("Example document keys:", sorted(example.keys()))
    print("Example sensor/imu_location/subject:", example.get("sensor"), example.get("imu_location"), example.get("subject"))

segments_df = pd.DataFrame([
    {
        "activity_label": doc["activity_label"],
        "activity_id": doc["activity_id"],
        "subject": doc["subject"],
        "imu_location": doc["imu_location"],
        "sensor": doc["sensor"],
        "segment_length": len(doc["data"]["acc_x"]),
        "sr": doc.get("sr", 100),
    }
    for doc in documents
])
segments_df["duration_sec"] = segments_df["segment_length"] / segments_df["sr"]
segments_df.head()

## Explore the nature of the data

Suggested exploratory plots for the PAMAP2 instances you loaded:

* Total recording time per activity (sum of segment lengths in seconds, grouped by `activity_label`).
* A time-domain plot of one segment per activity, so you can see the signal shape of each class.
* The distribution of segment counts per `(subject, activity_label)` pair — this exposes the class imbalance you will need to address.

In [ ]:
# Total recording time per activity
activity_duration = segments_df.groupby("activity_label")["duration_sec"].sum().sort_values(ascending=False)
plt.figure(figsize=(12, 5))
sns.barplot(x=activity_duration.index, y=activity_duration.values, hue=activity_duration.index, palette="viridis", legend=False)
plt.xticks(rotation=45, ha="right")
plt.title("Total recording time per activity - Protocol hand AccGyr")
plt.ylabel("Duration (s)")
plt.xlabel("Activity")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "activity_duration_protocol_hand_accgyr.png", bbox_inches="tight")
plt.show()

In [ ]:
# Time-domain plot of one segment per activity (first samples only for readability)
activity_examples = {}
for doc in documents:
    label = doc["activity_label"]
    if label not in activity_examples:
        activity_examples[label] = doc
    if len(activity_examples) == segments_df["activity_label"].nunique():
        break

plot_samples = 2000
for activity_label, doc in activity_examples.items():
    plt.figure(figsize=(16, 6))
    plot_instance_time_domain(pd.DataFrame(doc["data"]).head(plot_samples))
    plt.title(f"Time-domain segment for activity: {activity_label} (subject {doc['subject']}, first {plot_samples} samples)")
    plt.tight_layout()
    safe_label = activity_label.lower().replace(" ", "_").replace("/", "_")
    plt.savefig(RESULTS_DIR / f"time_domain_{safe_label}_protocol_hand_accgyr.png", bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
# Segment counts per activity and subject
plt.figure(figsize=(14, 6))
sns.countplot(data=segments_df, x="activity_label", hue="subject", palette="tab10")
plt.xticks(rotation=45, ha="right")
plt.title("Segment counts per activity and subject - Protocol hand AccGyr")
plt.ylabel("Segment count")
plt.xlabel("Activity")
plt.legend(title="Subject", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "segment_counts_protocol_hand_accgyr.png", bbox_inches="tight")
plt.show()

## Data Processing

* Apply the sliding window algorithm to each segment (use `sliding_window_pd` from `utils.py` with the parameters defined in `config.yml`).
* Apply a low-pass Butterworth filter to each window (use `apply_filter` / `filter_instances` from `utils.py`).
* Detect outliers and confirm that no `NaN` values survive from the ingestion step.

Keep `(window, activity_label, subject)` triplets together throughout this section — you need the subject ID for the train/test split.

In [ ]:
# Create sliding windows for each contiguous segment.
window_size = config["sliding_window"]["ws"]
overlap_ratio = config["sliding_window"]["overlap"]
step = int(window_size * (1 - overlap_ratio))
if step <= 0:
    step = window_size
print(f"Window size: {window_size}; step size: {step}; overlap ratio: {overlap_ratio}")

windowed_instances = []
for doc in tqdm_notebook(documents, desc="Creating sliding windows"):
    df_segment = pd.DataFrame(doc["data"])
    windows = sliding_window_pd(
        df_segment,
        ws=window_size,
        overlap=step,
        w_type=config["sliding_window"]["w_type"],
        w_center=config["sliding_window"]["w_center"],
        print_stats=False,
    )
    for window in windows:
        windowed_instances.append({
            "window": window,
            "activity_label": doc["activity_label"],
            "activity_id": doc["activity_id"],
            "subject": doc["subject"],
        })

print(f"Created {len(windowed_instances)} windows from {len(documents)} segments.")

In [ ]:
# Apply filtering to each window and store the filtered windows in a new list
filtered_windows = filter_instances(
    [item["window"] for item in windowed_instances],
    order=config["filter"]["order"],
    wn=config["filter"]["wn"],
    filter_type=config["filter"]["type"],
)

# Combine filtered windows with their corresponding labels and subjects into a new list of instances
filtered_instances = [
    {
        "window": filtered_windows[i],
        "activity_label": windowed_instances[i]["activity_label"],
        "activity_id": windowed_instances[i]["activity_id"],
        "subject": windowed_instances[i]["subject"],
    }
    for i in range(len(filtered_windows))
]
print(f"Filtered {len(filtered_instances)} windows.")

In [ ]:
# Check for NaN values in the filtered windows
nan_after_filter = [inst["window"].isna().sum().sum() for inst in filtered_instances]
print("NaN values per filtered window (sample):", nan_after_filter[:10])
assert not any(nan_after_filter), "There are NaN values after filtering!"

processed_instances = pd.DataFrame([
    {
        "activity_label": inst["activity_label"],
        "activity_id": inst["activity_id"],
        "subject": inst["subject"],
        "window": inst["window"],
    }
    for inst in filtered_instances
])
print(processed_instances.shape)

plt.figure(figsize=(12, 5))
sns.countplot(data=processed_instances, x="activity_label", hue="activity_label", palette="viridis", legend=False)
plt.xticks(rotation=45, ha="right")
plt.title("Window count per activity after segmentation - Protocol hand AccGyr")
plt.ylabel("Window count")
plt.xlabel("Activity")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "window_counts_protocol_hand_accgyr.png", bbox_inches="tight")
plt.show()

## Feature Engineering

This section builds the engineered-feature dataset required by the assignment. Each filtered window is summarized with time-domain and frequency-domain features, then the same subject-disjoint train/test split is reused to compare:

* raw flattened windows
* all engineered features
* selected engineered features

In [ ]:
# Feature engineering helpers.
def compute_magnitude(df):
    df = df.copy()
    prefixes = [""]
    prefixes.extend(sorted({col.rsplit("acc_x", 1)[0] for col in df.columns if col.endswith("acc_x")}))
    prefixes.extend(sorted({col.rsplit("gyr_x", 1)[0] for col in df.columns if col.endswith("gyr_x")}))
    prefixes = sorted(set(prefixes), key=lambda value: (len(value), value))

    for prefix in prefixes:
        acc_cols = [f"{prefix}acc_x", f"{prefix}acc_y", f"{prefix}acc_z"]
        gyr_cols = [f"{prefix}gyr_x", f"{prefix}gyr_y", f"{prefix}gyr_z"]
        if all(col in df.columns for col in acc_cols):
            df[f"{prefix}acc_magnitude"] = np.sqrt(np.sum(df[acc_cols] ** 2, axis=1))
        if all(col in df.columns for col in gyr_cols):
            df[f"{prefix}gyr_magnitude"] = np.sqrt(np.sum(df[gyr_cols] ** 2, axis=1))
    return df


def zero_crossing_rate(series):
    values = series.to_numpy()
    return np.mean(values[1:] * values[:-1] < 0)


def extract_time_features(df):
    features = {}
    for col in df.columns:
        s = df[col]
        features[f"{col}_mean"] = s.mean()
        features[f"{col}_median"] = s.median()
        features[f"{col}_std"] = s.std()
        features[f"{col}_min"] = s.min()
        features[f"{col}_max"] = s.max()
        features[f"{col}_iqr"] = s.quantile(0.75) - s.quantile(0.25)
        features[f"{col}_energy"] = np.mean(np.square(s))
        features[f"{col}_skew"] = s.skew()
        features[f"{col}_kurtosis"] = s.kurtosis()
        features[f"{col}_zcr"] = zero_crossing_rate(s)
    return pd.Series(features)


def extract_frequency_features(df, sr=100):
    freq_features = {}
    for col in df.columns:
        values = df[col].to_numpy()
        fft_vals = np.fft.rfft(values)
        fft_freqs = np.fft.rfftfreq(len(values), d=1 / sr)
        fft_magnitude = np.abs(fft_vals)
        magnitude_sum = fft_magnitude.sum()

        if magnitude_sum > 0:
            centroid = np.sum(fft_freqs * fft_magnitude) / magnitude_sum
            bandwidth = np.sqrt(np.sum(((fft_freqs - centroid) ** 2) * fft_magnitude) / magnitude_sum)
            cumulative_energy = np.cumsum(fft_magnitude)
            rolloff_freq = fft_freqs[np.searchsorted(cumulative_energy, 0.85 * cumulative_energy[-1])]
            dominant_freq = fft_freqs[np.argmax(fft_magnitude[1:]) + 1] if len(fft_magnitude) > 1 else 0
        else:
            centroid = bandwidth = rolloff_freq = dominant_freq = 0

        freq_features[f"{col}_spectral_centroid"] = centroid
        freq_features[f"{col}_spectral_bandwidth"] = bandwidth
        freq_features[f"{col}_spectral_rolloff"] = rolloff_freq
        freq_features[f"{col}_dominant_freq"] = dominant_freq
    return pd.Series(freq_features)


def extract_window_features(window, sr=100):
    window_with_mag = compute_magnitude(window)
    return pd.concat([
        extract_time_features(window_with_mag),
        extract_frequency_features(window_with_mag, sr=sr),
    ])

# Add magnitudes to the raw-window representation too, so raw and engineered pipelines use the same channels.
processed_instances["window"] = processed_instances["window"].apply(compute_magnitude)

feature_rows = []
for _, inst in tqdm_notebook(processed_instances.iterrows(), total=len(processed_instances), desc="Extracting engineered features"):
    features = extract_window_features(inst["window"], sr=100)
    features["activity_id"] = inst["activity_id"]
    features["activity_label"] = inst["activity_label"]
    features["subject"] = inst["subject"]
    feature_rows.append(features)

feature_df = pd.DataFrame(feature_rows).replace([np.inf, -np.inf], np.nan).dropna()
print("Engineered feature table shape:", feature_df.shape)
feature_df.head()

## Train/Test split

The split must be **by subject**: 6–7 subjects in train, 2–3 in test, with **no overlap**. This forces the model to generalize to users it has never seen, which is the realistic deployment scenario.

Do **not** use `train_test_split` with random shuffling — that would leak windows from the same subject into both sets and inflate the reported metrics.

In [ ]:
# Train-test split by subject 

X_train, y_train = [], []
X_test, y_test = [], []


In [ ]:
TRAIN_SUBJECTS = ["101", "102", "103", "104", "105", "107"]
TEST_SUBJECTS  = ["106", "108", "109"]

assert set(TRAIN_SUBJECTS).isdisjoint(TEST_SUBJECTS)

train_labels = set(processed_instances.loc[processed_instances["subject"].isin(TRAIN_SUBJECTS), "activity_id"])
test_labels = set(processed_instances.loc[processed_instances["subject"].isin(TEST_SUBJECTS), "activity_id"])
print("Train labels:", sorted(train_labels))
print("Test labels:", sorted(test_labels))
print("Labels missing from train:", sorted(test_labels - train_labels))
print("Labels missing from test:", sorted(train_labels - test_labels))

In [ ]:
# Raw/windowed time-series representation: flatten each filtered window.
X_train_raw, y_train_raw = [], []
X_test_raw, y_test_raw = [], []

for _, inst in processed_instances.iterrows():
    subject = inst["subject"]
    features = inst["window"].to_numpy().flatten()
    label = inst["activity_id"]

    if subject in TRAIN_SUBJECTS:
        X_train_raw.append(features)
        y_train_raw.append(label)
    elif subject in TEST_SUBJECTS:
        X_test_raw.append(features)
        y_test_raw.append(label)

X_train_raw = np.vstack(X_train_raw)
X_test_raw = np.vstack(X_test_raw)
y_train_raw = np.array(y_train_raw)
y_test_raw = np.array(y_test_raw)

# Keep the old variable names for the raw/PCA baseline cells below.
X_train, X_test = X_train_raw, X_test_raw
y_train, y_test = y_train_raw, y_test_raw

print("Raw train shape:", X_train_raw.shape)
print("Raw test shape:", X_test_raw.shape)

## Scaling

Fit the scaler on the training subjects only, then apply it to the test subjects.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [ ]:
# Scaling the features using StandardScaler (zero mean, unit variance)
scaler = StandardScaler()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

# Fit the scaler on the training data and transform both training and test data
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Scaled feature shape: {X_train.shape}")

## Dimensionality Reduction (PCA)

Apply PCA to reduce the feature dimensionality while retaining most of the variance. This is especially useful for high-dimensional windowed sensor data.



In [ ]:
from sklearn.decomposition import PCA

# Apply PCA with 99% variance retention (or adjust n_components based on your analysis)
n_components = config["PCA"]["n_comp"]  

print(f"\nApplying PCA with {n_components} components (99% variance retention)...")

pca = PCA(n_components=n_components, random_state=42)

# Fit PCA on training data only (avoid data leakage)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"PCA-reduced feature shape: {X_train_pca.shape}")
print(f"Dimensionality reduction: {X_train.shape[1]} → {X_train_pca.shape[1]} features ({100*X_train_pca.shape[1]/X_train.shape[1]:.1f}%)")
print(f"Variance retained: {pca.explained_variance_ratio_.sum():.4f}")


In [ ]:
from sklearn.decomposition import PCA

# Analyze explained variance with all components
pca_full = PCA()
pca_full.fit(X_train)

# Cumulative explained variance
cumsum_var = np.cumsum(pca_full.explained_variance_ratio_)

# Plot explained variance
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, len(pca_full.explained_variance_ratio_) + 1), 
         pca_full.explained_variance_ratio_, 'bo-', linewidth=2)
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Scree Plot: Explained Variance per Component")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumsum_var) + 1), cumsum_var, 'ro-', linewidth=2)
plt.axhline(y=0.85, color='g', linestyle='--', label='85% variance')
plt.axhline(y=0.95, color='b', linestyle='--', label='95% variance')
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Explained Variance")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Determine optimal number of components to retain 95% variance
n_components_95 = np.argmax(cumsum_var >= 0.95) + 1
n_components_85 = np.argmax(cumsum_var >= 0.85) + 1

print(f"Original feature dimension: {X_train.shape[1]}")
print(f"Components to retain 85% variance: {n_components_85}")
print(f"Components to retain 95% variance: {n_components_95}")
print(f"Variance retained with {n_components_95} components: {cumsum_var[n_components_95-1]:.4f}")


### Apply simple classifier

In [ ]:
# Use PCA-reduced features
print(f"Using PCA-Reduced Features: {X_train_pca.shape[1]} features")

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train classifiers on PCA-reduced features

# Hyperparameters from config for SVC
kernel = config["classifier"]["SVC"]["kernel"]
C = config["classifier"]["SVC"]["C"]
gamma = config["classifier"]["SVC"]["gamma"]
class_weight = config["classifier"]["SVC"].get("class_weight", None)

# Hyperparameters from config for Random Forest
n_estimators = config["classifier"]["RandomForest"]["n_estimators"]
max_depth = config["classifier"]["RandomForest"]["max_depth"]

print("Training on PCA-Reduced Features...")

# SVM classifier
print("Training SVC:")
svm = SVC(kernel=kernel, C=C, gamma=gamma, class_weight=class_weight, random_state=42)
svm.fit(X_train_pca, y_train)

# Random Forest classifier
print("Training Random Forest:")
rf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
rf.fit(X_train_pca, y_train)


### Evaluate simple classifier

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

summary_results = []

def evaluate_model(name, model, X_test_eval, y_test_eval, result_rows, labels=None, save_cm=None):
    y_pred = model.predict(X_test_eval)
    acc = accuracy_score(y_test_eval, y_pred)
    precision = precision_score(y_test_eval, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test_eval, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test_eval, y_pred, average="weighted", zero_division=0)

    print(f"\n{name}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(classification_report(y_test_eval, y_pred, zero_division=0))

    result_rows.append({
        "Approach": name,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
    })

    if save_cm is not None:
        disp = ConfusionMatrixDisplay.from_predictions(y_test_eval, y_pred, display_labels=labels, xticks_rotation=45, cmap="Blues")
        disp.ax_.set_title(name)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / save_cm, bbox_inches="tight")
        plt.show()

    return y_pred

print("\nEVALUATION: Raw Windowed PCA Models")
print("=" * 60)
raw_label_order = sorted(np.unique(np.concatenate([y_train, y_test])))

evaluate_model("Raw PCA SVC", svm, X_test_pca, y_test, summary_results, labels=raw_label_order, save_cm="cm_raw_pca_svc.png")
evaluate_model("Raw PCA Random Forest", rf, X_test_pca, y_test, summary_results, labels=raw_label_order, save_cm="cm_raw_pca_random_forest.png")

results_df = pd.DataFrame(summary_results)
print("\nSUMMARY TABLE")
print(results_df.to_string(index=False))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
from sklearn.metrics import classification_report

## Engineered-Feature Modeling

Train and evaluate models on the extracted feature table. This satisfies the full-feature comparison required by the assignment.

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

metadata_cols = ["activity_id", "activity_label", "subject"]
feature_cols = [col for col in feature_df.columns if col not in metadata_cols]

train_mask = feature_df["subject"].isin(TRAIN_SUBJECTS)
test_mask = feature_df["subject"].isin(TEST_SUBJECTS)

X_train_features = feature_df.loc[train_mask, feature_cols]
X_test_features = feature_df.loc[test_mask, feature_cols]
y_train_features = feature_df.loc[train_mask, "activity_id"].to_numpy()
y_test_features = feature_df.loc[test_mask, "activity_id"].to_numpy()

feature_scaler = StandardScaler()
X_train_features_scaled = feature_scaler.fit_transform(X_train_features)
X_test_features_scaled = feature_scaler.transform(X_test_features)

print("All engineered features train shape:", X_train_features_scaled.shape)
print("All engineered features test shape:", X_test_features_scaled.shape)

svc_features = SVC(kernel=kernel, C=C, gamma=gamma, class_weight="balanced", random_state=42)
rf_features = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, class_weight="balanced_subsample", random_state=42)

svc_features.fit(X_train_features_scaled, y_train_features)
rf_features.fit(X_train_features_scaled, y_train_features)

feature_label_order = sorted(np.unique(np.concatenate([y_train_features, y_test_features])))
evaluate_model("All Engineered Features SVC", svc_features, X_test_features_scaled, y_test_features, summary_results, labels=feature_label_order, save_cm="cm_all_features_svc.png")
evaluate_model("All Engineered Features Random Forest", rf_features, X_test_features_scaled, y_test_features, summary_results, labels=feature_label_order, save_cm="cm_all_features_random_forest.png")

## Feature Selection

Select a smaller subset of the engineered features with mutual information, then train/evaluate the same models again.

In [ ]:
k_features = min(30, X_train_features_scaled.shape[1])
selector = SelectKBest(score_func=mutual_info_classif, k=k_features)
X_train_selected = selector.fit_transform(X_train_features_scaled, y_train_features)
X_test_selected = selector.transform(X_test_features_scaled)
selected_features = X_train_features.columns[selector.get_support()].tolist()

print(f"Selected {len(selected_features)} features:")
for feature in selected_features:
    print("-", feature)

scores = pd.DataFrame({
    "feature": X_train_features.columns,
    "score": selector.scores_,
}).sort_values("score", ascending=False).head(k_features)

plt.figure(figsize=(10, 8))
sns.barplot(data=scores, y="feature", x="score", hue="feature", palette="viridis", legend=False)
plt.title("Top Selected Engineered Features")
plt.xlabel("Mutual information score")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "selected_feature_scores.png", bbox_inches="tight")
plt.show()

svc_selected = SVC(kernel=kernel, C=C, gamma=gamma, class_weight="balanced", random_state=42)
rf_selected = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, class_weight="balanced_subsample", random_state=42)

svc_selected.fit(X_train_selected, y_train_features)
rf_selected.fit(X_train_selected, y_train_features)

evaluate_model("Selected Engineered Features SVC", svc_selected, X_test_selected, y_test_features, summary_results, labels=feature_label_order, save_cm="cm_selected_features_svc.png")
evaluate_model("Selected Engineered Features Random Forest", rf_selected, X_test_selected, y_test_features, summary_results, labels=feature_label_order, save_cm="cm_selected_features_random_forest.png")

final_results_df = pd.DataFrame(summary_results).sort_values("F1-Score", ascending=False)
print("\nFINAL COMPARISON")
print(final_results_df.to_string(index=False))
final_results_df.to_csv(RESULTS_DIR / "model_comparison_protocol_hand_accgyr.csv", index=False)

## Combined Wrist + Chest + Ankle Model

This experiment concatenates the three IMU locations into one multi-sensor window, then trains SVC and Random Forest on engineered features. It answers how performance changes when all available inertial sensors are used together.

In [ ]:
RUN_COMBINED_SENSOR_MODEL = True


def load_combined_imu_documents():
    sensor_docs = list(coll.find({"split": "Protocol", "sensor": "AccGyr"}))
    docs_by_key = {}
    for doc in sensor_docs:
        key = (
            doc["subject"],
            doc.get("segment_index"),
            doc["activity_id"],
            doc["activity_label"],
        )
        docs_by_key.setdefault(key, {})[doc["imu_location"]] = doc

    combined_documents = []
    for (subject, segment_index, activity_id, activity_label), location_docs in docs_by_key.items():
        if not {"hand", "chest", "ankle"}.issubset(location_docs):
            continue

        min_len = min(len(location_docs[loc]["data"]["acc_x"]) for loc in ["hand", "chest", "ankle"])
        combined_data = {}
        for loc in ["hand", "chest", "ankle"]:
            for axis_name, values in location_docs[loc]["data"].items():
                combined_data[f"{loc}_{axis_name}"] = values[:min_len]

        combined_documents.append({
            "data": combined_data,
            "activity_id": activity_id,
            "activity_label": activity_label,
            "subject": subject,
            "segment_index": segment_index,
            "sr": 100,
            "segment_length": min_len,
        })

    return combined_documents


def build_combined_processed_instances(combined_documents):
    combined_windowed = []
    for doc in tqdm_notebook(combined_documents, desc="Windowing combined IMUs"):
        df_segment = pd.DataFrame(doc["data"])
        windows = sliding_window_pd(
            df_segment,
            ws=window_size,
            overlap=step,
            w_type=config["sliding_window"]["w_type"],
            w_center=config["sliding_window"]["w_center"],
            print_stats=False,
        )
        for window in windows:
            combined_windowed.append({
                "window": window,
                "activity_label": doc["activity_label"],
                "activity_id": doc["activity_id"],
                "subject": doc["subject"],
            })

    combined_filtered_windows = filter_instances(
        [item["window"] for item in combined_windowed],
        order=config["filter"]["order"],
        wn=config["filter"]["wn"],
        filter_type=config["filter"]["type"],
    )

    return pd.DataFrame([
        {
            "activity_label": combined_windowed[i]["activity_label"],
            "activity_id": combined_windowed[i]["activity_id"],
            "subject": combined_windowed[i]["subject"],
            "window": compute_magnitude(combined_filtered_windows[i]),
        }
        for i in range(len(combined_filtered_windows))
    ])


def build_feature_table_from_processed(processed_df, desc):
    rows = []
    for _, inst in tqdm_notebook(processed_df.iterrows(), total=len(processed_df), desc=desc):
        features = extract_window_features(inst["window"], sr=100)
        features["activity_id"] = inst["activity_id"]
        features["activity_label"] = inst["activity_label"]
        features["subject"] = inst["subject"]
        rows.append(features)
    return pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan).dropna()


if RUN_COMBINED_SENSOR_MODEL:
    combined_documents = load_combined_imu_documents()
    print(f"Combined Protocol segments: {len(combined_documents)}")

    combined_processed = build_combined_processed_instances(combined_documents)
    print("Combined processed windows:", combined_processed.shape)

    combined_feature_df = build_feature_table_from_processed(combined_processed, "Features combined IMUs")
    combined_feature_cols = [col for col in combined_feature_df.columns if col not in metadata_cols]

    train_mask_combined = combined_feature_df["subject"].isin(TRAIN_SUBJECTS)
    test_mask_combined = combined_feature_df["subject"].isin(TEST_SUBJECTS)

    X_train_combined = combined_feature_df.loc[train_mask_combined, combined_feature_cols]
    X_test_combined = combined_feature_df.loc[test_mask_combined, combined_feature_cols]
    y_train_combined = combined_feature_df.loc[train_mask_combined, "activity_id"].to_numpy()
    y_test_combined = combined_feature_df.loc[test_mask_combined, "activity_id"].to_numpy()

    combined_scaler = StandardScaler()
    X_train_combined_scaled = combined_scaler.fit_transform(X_train_combined)
    X_test_combined_scaled = combined_scaler.transform(X_test_combined)

    combined_svc = SVC(kernel=kernel, C=C, gamma=gamma, class_weight="balanced", random_state=42)
    combined_rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        class_weight="balanced_subsample",
        random_state=42,
    )

    combined_svc.fit(X_train_combined_scaled, y_train_combined)
    combined_rf.fit(X_train_combined_scaled, y_train_combined)

    combined_results = []
    combined_label_order = sorted(np.unique(np.concatenate([y_train_combined, y_test_combined])))
    evaluate_model(
        "Combined Wrist+Chest+Ankle Engineered Features SVC",
        combined_svc,
        X_test_combined_scaled,
        y_test_combined,
        combined_results,
        labels=combined_label_order,
        save_cm="cm_combined_all_imus_svc.png",
    )
    evaluate_model(
        "Combined Wrist+Chest+Ankle Engineered Features Random Forest",
        combined_rf,
        X_test_combined_scaled,
        y_test_combined,
        combined_results,
        labels=combined_label_order,
        save_cm="cm_combined_all_imus_random_forest.png",
    )

    combined_results_df = pd.DataFrame(combined_results).sort_values("F1-Score", ascending=False)
    print("\nCOMBINED SENSOR MODEL RESULTS")
    print(combined_results_df.to_string(index=False))
    combined_results_df.to_csv(RESULTS_DIR / "combined_wrist_chest_ankle_model_comparison.csv", index=False)
else:
    print("Combined sensor model skipped. Set RUN_COMBINED_SENSOR_MODEL = True to run it.")

## Sensor Configuration Comparison

The wrist configuration is the baseline. In PAMAP2 this IMU is stored as `hand`. This section repeats the engineered-feature Random Forest pipeline for hand, chest, and ankle IMUs so the report can compare sensor placement trade-offs.

In [ ]:
RUN_SENSOR_COMPARISON = False


def build_processed_instances_for_sensor(imu_location):
    sensor_query = {"split": "Protocol", "imu_location": imu_location, "sensor": "AccGyr"}
    sensor_documents = list(coll.find(sensor_query))
    print(f"\n{imu_location}: loaded {len(sensor_documents)} Protocol segments")

    sensor_windowed_instances = []
    for doc in tqdm_notebook(sensor_documents, desc=f"Windowing {imu_location}"):
        df_segment = pd.DataFrame(doc["data"])
        windows = sliding_window_pd(
            df_segment,
            ws=window_size,
            overlap=step,
            w_type=config["sliding_window"]["w_type"],
            w_center=config["sliding_window"]["w_center"],
            print_stats=False,
        )
        for window in windows:
            sensor_windowed_instances.append({
                "window": window,
                "activity_label": doc["activity_label"],
                "activity_id": doc["activity_id"],
                "subject": doc["subject"],
            })

    sensor_filtered_windows = filter_instances(
        [item["window"] for item in sensor_windowed_instances],
        order=config["filter"]["order"],
        wn=config["filter"]["wn"],
        filter_type=config["filter"]["type"],
    )

    sensor_processed = pd.DataFrame([
        {
            "activity_label": sensor_windowed_instances[i]["activity_label"],
            "activity_id": sensor_windowed_instances[i]["activity_id"],
            "subject": sensor_windowed_instances[i]["subject"],
            "window": compute_magnitude(sensor_filtered_windows[i]),
        }
        for i in range(len(sensor_filtered_windows))
    ])
    print(f"{imu_location}: processed windows {sensor_processed.shape}")
    return sensor_processed


def build_feature_table_for_sensor(sensor_processed, imu_location):
    rows = []
    for _, inst in tqdm_notebook(sensor_processed.iterrows(), total=len(sensor_processed), desc=f"Features {imu_location}"):
        features = extract_window_features(inst["window"], sr=100)
        features["activity_id"] = inst["activity_id"]
        features["activity_label"] = inst["activity_label"]
        features["subject"] = inst["subject"]
        rows.append(features)
    return pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan).dropna()


def evaluate_sensor_configuration(imu_location):
    sensor_processed = build_processed_instances_for_sensor(imu_location)
    sensor_features = build_feature_table_for_sensor(sensor_processed, imu_location)

    feature_cols_sensor = [col for col in sensor_features.columns if col not in metadata_cols]
    train_mask_sensor = sensor_features["subject"].isin(TRAIN_SUBJECTS)
    test_mask_sensor = sensor_features["subject"].isin(TEST_SUBJECTS)

    X_train_sensor = sensor_features.loc[train_mask_sensor, feature_cols_sensor]
    X_test_sensor = sensor_features.loc[test_mask_sensor, feature_cols_sensor]
    y_train_sensor = sensor_features.loc[train_mask_sensor, "activity_id"].to_numpy()
    y_test_sensor = sensor_features.loc[test_mask_sensor, "activity_id"].to_numpy()

    scaler_sensor = StandardScaler()
    X_train_sensor_scaled = scaler_sensor.fit_transform(X_train_sensor)
    X_test_sensor_scaled = scaler_sensor.transform(X_test_sensor)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        class_weight="balanced_subsample",
        random_state=42,
    )
    model.fit(X_train_sensor_scaled, y_train_sensor)

    rows = []
    labels = sorted(np.unique(np.concatenate([y_train_sensor, y_test_sensor])))
    evaluate_model(
        f"{imu_location.title()} All Engineered Features Random Forest",
        model,
        X_test_sensor_scaled,
        y_test_sensor,
        rows,
        labels=labels,
        save_cm=f"cm_sensor_{imu_location}_all_features_random_forest.png",
    )
    rows[0]["Sensor"] = imu_location
    rows[0]["Train Windows"] = len(y_train_sensor)
    rows[0]["Test Windows"] = len(y_test_sensor)
    rows[0]["Features"] = len(feature_cols_sensor)
    return rows[0]


if RUN_SENSOR_COMPARISON:
    sensor_results = []
    for imu_location in ["hand", "chest", "ankle"]:
        sensor_results.append(evaluate_sensor_configuration(imu_location))

    sensor_results_df = pd.DataFrame(sensor_results).sort_values("F1-Score", ascending=False)
    print("\nSENSOR CONFIGURATION COMPARISON")
    print(sensor_results_df.to_string(index=False))
    sensor_results_df.to_csv(RESULTS_DIR / "sensor_comparison_protocol_accgyr_random_forest.csv", index=False)

    plt.figure(figsize=(8, 5))
    sns.barplot(data=sensor_results_df, x="Sensor", y="F1-Score", hue="Sensor", palette="viridis", legend=False)
    plt.ylim(0, 1)
    plt.title("Sensor Configuration Comparison - Engineered Features Random Forest")
    plt.ylabel("Weighted F1-score")
    plt.xlabel("IMU location")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "sensor_comparison_protocol_accgyr_random_forest.png", bbox_inches="tight")
    plt.show()
else:
    print("Sensor comparison skipped. Set RUN_SENSOR_COMPARISON = True to run wrist/chest/ankle comparison.")

### Apply optimization with Grid Search and/or Cross-validation

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

In [ ]:
def run_grid_search(name, model, param_grid, X_train, y_train, cv, verbose):
    print(f"\n GRID SEARCH: {name}\n")

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=config["fine_tune"]["grid_search"]["scoring"],
        refit=True,
        cv=cv,
        verbose=verbose,
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    print("\nBEST RESULT")
    print("Params:", grid.best_params_)
    print("F1:", round(grid.best_score_, 4))

    return grid


### Evaluate optimized classifier

Once you are satisfied with the wrist-only configuration, repeat the **load → process → train → evaluate** flow for the chest and ankle IMUs (and combinations thereof) and compare the metrics in your report.

In [ ]:
RUN_GRID_SEARCH = False

if RUN_GRID_SEARCH:
    print("Evaluation of Grid Search results on test set:")

    grid = run_grid_search(
        name="SVC",
        model=SVC(random_state=42),
        param_grid=config["fine_tune"]["SVC"]["param_grid"],
        X_train=X_train_pca,
        y_train=y_train,
        cv=config["fine_tune"]["SVC"]["cv"],
        verbose=config["fine_tune"]["SVC"]["verbose"],
    )

    best_svc = SVC(**grid.best_params_, random_state=42)
    best_svc.fit(X_train_pca, y_train)
    y_pred_svc = best_svc.predict(X_test_pca)

    print("\nClassification Report for Best SVC Model:")
    print(classification_report(y_test, y_pred_svc, zero_division=0))

    cm_svc = confusion_matrix(y_test, y_pred_svc)
    print("Confusion Matrix for Best SVC Model:")
    print(cm_svc)

    disp_svc = ConfusionMatrixDisplay(confusion_matrix=cm_svc)
    disp_svc.plot(cmap=plt.cm.Blues)
    plt.title("Confusion Matrix - Best SVC Model")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "cm_grid_best_svc.png", bbox_inches="tight")
    plt.show()

    grid = run_grid_search(
        name="RandomForest",
        model=RandomForestClassifier(random_state=42),
        param_grid=config["fine_tune"]["RandomForest"]["param_grid"],
        X_train=X_train_pca,
        y_train=y_train,
        cv=config["fine_tune"]["RandomForest"]["cv"],
        verbose=config["fine_tune"]["RandomForest"]["verbose"],
    )

    best_rf = RandomForestClassifier(**grid.best_params_, random_state=42)
    best_rf.fit(X_train_pca, y_train)
    y_pred_rf = best_rf.predict(X_test_pca)

    print("\nClassification Report for Best Random Forest Model:")
    print(classification_report(y_test, y_pred_rf, zero_division=0))

    cm_rf = confusion_matrix(y_test, y_pred_rf)
    print("Confusion Matrix for Best Random Forest Model:")
    print(cm_rf)

    disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf)
    disp_rf.plot(cmap=plt.cm.Oranges)
    plt.title("Confusion Matrix - Best Random Forest Model")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "cm_grid_best_random_forest.png", bbox_inches="tight")
    plt.show()
else:
    print("Grid Search skipped. Set RUN_GRID_SEARCH = True when the baseline and feature pipelines are stable.")